# T12 - 理财产品权益配置 - 重构Notebook

## 项目概述

本项目专注于分析银行理财产品中的权益资产配置情况，研究理财资金的权益资产暴露程度、配置策略、风险控制措施等，为理财投资决策提供参考。

### 功能描述
- 产品分析：按风险等级、投资类型、期限对理财产品进行分类
- 权益配置分析：分析权益资产的配置比例和类型分布
- 收益分析：计算产品收益率、年化收益、波动率等指标
- 风险分析：计算VaR、CVaR、最大回撤、夏普比率等风险指标

### 理财产品概述
- **风险等级**：R1（低风险）到R5（高风险）
- **投资范围**：固定收益类、权益类、混合类
- **投资期限**：短期（<3个月）、中期（3-12个月）、长期（>12个月）

### 监管要求
1. **理财新规**：资管新规对理财产品的规范
2. **净值化管理**：理财产品需净值化
3. **信息披露**：定期披露产品运作情况
4. **风险揭示**：充分揭示投资风险

---
## 1. 环境配置

导入依赖库并配置参数。

In [ ]:
# 1.1 导入依赖
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine

# 配置中文字体
plt.rcParams['font.sans-serif'] = ['Noto Sans CJK SC', 'WenQuanYi Micro Hei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

print("依赖导入完成")

In [ ]:
# 1.2 配置参数

# 数据库配置（使用环境变量）
DB_HOST = os.environ.get('DB_HOST', 'localhost')
DB_PORT = os.environ.get('DB_PORT', '3306')
DB_USER = os.environ.get('DB_USER', 'root')
DB_PASSWORD = os.environ.get('DB_PASSWORD', '')
DB_NAME = os.environ.get('DB_NAME', 'wealth_products')

# 产品风险等级配置
RISK_CLASSES = {
    'R1': '低风险',
    'R2': '中低风险',
    'R3': '中等风险',
    'R4': '中高风险',
    'R5': '高风险'
}

# 投资类型配置
INVESTMENT_TYPES = {
    'fixed_income': '固定收益类',
    'equity': '权益类',
    'mixed': '混合类'
}

# 期限分类
MATURITIES = {
    'short': '短期（<3个月）',
    'medium': '中期（3-12个月）',
    'long': '长期（>12个月）'
}

# 权益资产类型
EQUITY_TYPES = {
    'stock': '股票',
    'stock_index': '股票指数',
    'equity_fund': '股票基金',
    'stock_option': '股票期权',
    'equity_derivatives': '权益衍生品'
}

# 权益配置比例限制
WEALTH_PRODUCT_CONFIG = {
    'min_equity_ratio': 0,      # 最小权益比例
    'max_equity_ratio': 0.8,    # 最大权益比例
    'risk_weight': {'stock': 0.5, 'stock_index': 0.3, 'equity_fund': 0.2},
    'liquidity_requirement': 0.1,  # 流动性要求（10%）
    'leverage_limit': 1.2       # 杠杆限制
}

print("配置参数加载完成")

---
## 2. 数据获取

获取产品信息、净值数据、资产配置数据。

In [ ]:
# 2.1 数据库连接工具

def get_database_engine():
    """创建数据库连接引擎"""
    connection_string = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
    engine = create_engine(connection_string, pool_recycle=3600)
    return engine

def execute_query(query, engine=None):
    """执行SQL查询并返回DataFrame"""
    if engine is None:
        engine = get_database_engine()
    return pd.read_sql(query, engine)

print("数据库工具函数定义完成")

In [ ]:
# 2.2 获取产品信息

def get_product_info(engine=None):
    """获取理财产品基本信息"""
    query = """
    SELECT 
        product_id,
        product_name,
        risk_level,
        investment_type,
        maturity_type,
        issue_date,
        maturity_date,
        nav
    FROM wealth_products
    WHERE status = 'active'
    """
    return execute_query(query, engine)

print("产品信息获取函数定义完成")

In [ ]:
# 2.3 获取净值数据

def get_nav_data(product_id=None, start_date=None, end_date=None, engine=None):
    """获取产品净值数据"""
    query = """
    SELECT 
        product_id,
        nav_date,
        nav,
        accumulated_nav
    FROM product_nav
    WHERE 1=1
    """
    
    if product_id:
        query += f" AND product_id = '{product_id}'"
    if start_date:
        query += f" AND nav_date >= '{start_date}'"
    if end_date:
        query += f" AND nav_date <= '{end_date}'"
    
    query += " ORDER BY product_id, nav_date"
    return execute_query(query, engine)

print("净值数据获取函数定义完成")

In [ ]:
# 2.4 获取资产配置数据

def get_allocation_data(product_id=None, report_date=None, engine=None):
    """获取产品资产配置数据"""
    query = """
    SELECT 
        product_id,
        report_date,
        asset_type,
        asset_value,
        total_value,
        allocation_ratio
    FROM product_allocation
    WHERE 1=1
    """
    
    if product_id:
        query += f" AND product_id = '{product_id}'"
    if report_date:
        query += f" AND report_date = '{report_date}'"
    
    return execute_query(query, engine)

print("资产配置数据获取函数定义完成")

In [ ]:
# 2.5 从Excel文件读取数据（示例）

def load_excel_data(file_path, sheet_name=None):
    """从Excel文件读取数据"""
    try:
        if sheet_name:
            df = pd.read_excel(file_path, sheet_name=sheet_name)
        else:
            df = pd.read_excel(file_path)
        print(f"成功读取数据，形状: {df.shape}")
        return df
    except Exception as e:
        print(f"读取Excel文件失败: {e}")
        return None

# 示例：读取波动率数据
# excel_file = '银行理财权益测算.xlsx'
# volatility_data = load_excel_data(excel_file, '年化波动率全量')

print("Excel数据读取函数定义完成")

---
## 3. 数据处理

数据清洗、产品分类、配置比例计算。

In [ ]:
# 3.1 数据清洗

def clean_product_data(df):
    """清洗产品数据"""
    # 删除重复记录
    df = df.drop_duplicates()
    
    # 处理缺失值
    df = df.dropna(subset=['product_id', 'risk_level'])
    
    # 标准化风险等级
    df['risk_level'] = df['risk_level'].str.upper().str.strip()
    
    return df

def clean_nav_data(df):
    """清洗净值数据"""
    # 删除重复记录
    df = df.drop_duplicates(subset=['product_id', 'nav_date'])
    
    # 按日期排序
    df = df.sort_values(['product_id', 'nav_date'])
    
    # 处理异常净值
    df = df[df['nav'] > 0]
    
    return df

print("数据清洗函数定义完成")

In [ ]:
# 3.2 产品分类

def classify_products(product_data):
    """产品分类"""
    # 按风险等级分类
    product_data['risk_class_name'] = product_data['risk_level'].map(RISK_CLASSES)
    
    # 按投资类型分类
    product_data['investment_type_name'] = product_data.get('investment_type', 'mixed').map(INVESTMENT_TYPES)
    
    # 按期限分类
    def get_maturity_type(row):
        if 'maturity_days' in row:
            days = row['maturity_days']
            if days < 90:
                return '短期（<3个月）'
            elif days <= 365:
                return '中期（3-12个月）'
            else:
                return '长期（>12个月）'
        return '中期（3-12个月）'
    
    product_data['maturity_name'] = product_data.apply(get_maturity_type, axis=1)
    
    return product_data

print("产品分类函数定义完成")

In [ ]:
# 3.3 配置比例计算

def calculate_allocation_ratio(allocation_data):
    """计算资产配置比例"""
    # 如果已有配置比例列，直接返回
    if 'allocation_ratio' in allocation_data.columns:
        return allocation_data
    
    # 计算配置比例
    allocation_data['allocation_ratio'] = allocation_data['asset_value'] / allocation_data['total_value']
    
    return allocation_data

print("配置比例计算函数定义完成")

---
## 4. 核心逻辑

产品分类、权益配置分析、收益计算、风险指标计算。

In [ ]:
# 4.1 权益配置分析

def analyze_equity_allocation(allocation_data):
    """分析权益配置比例"""
    # 筛选权益类资产
    equity_assets = allocation_data[allocation_data['asset_type'].isin(EQUITY_TYPES.keys())]
    
    # 计算整体权益占比
    equity_ratio = equity_assets['asset_value'].sum() / allocation_data['total_value'].sum()
    
    # 按产品类型分组
    if 'product_type' in allocation_data.columns:
        by_product_type = allocation_data.groupby('product_type').apply(
            lambda x: x[x['asset_type'].isin(EQUITY_TYPES.keys())]['asset_value'].sum() / x['total_value'].sum()
        )
    else:
        by_product_type = None
    
    # 按风险等级分组
    if 'risk_level' in allocation_data.columns:
        by_risk = allocation_data.groupby('risk_level').apply(
            lambda x: x[x['asset_type'].isin(EQUITY_TYPES.keys())]['asset_value'].sum() / x['total_value'].sum()
        )
    else:
        by_risk = None
    
    return {
        'overall_equity_ratio': equity_ratio,
        'by_product_type': by_product_type,
        'by_risk': by_risk
    }

print("权益配置分析函数定义完成")

In [ ]:
# 4.2 产品收益计算

def calculate_product_returns(product_nav_data):
    """计算产品收益率"""
    # 确保数据按日期排序
    product_nav_data = product_nav_data.sort_values('nav_date')
    
    # 日收益率
    daily_return = product_nav_data['nav'].pct_change()
    
    # 累计收益率
    cumulative_return = (1 + daily_return).cumprod() - 1
    
    # 年化收益率
    n_days = len(daily_return.dropna())
    if n_days > 0 and cumulative_return.iloc[-1] is not None:
        annualized_return = (1 + cumulative_return.iloc[-1]) ** (252 / n_days) - 1
    else:
        annualized_return = None
    
    # 波动率
    volatility = daily_return.std() * np.sqrt(252)
    
    return {
        'daily_return': daily_return,
        'cumulative_return': cumulative_return,
        'annualized_return': annualized_return,
        'volatility': volatility
    }

print("产品收益计算函数定义完成")

In [ ]:
# 4.3 风险指标计算（VaR/CVaR/夏普比率）

def calculate_risk_metrics(return_data, risk_free_rate=0.02):
    """计算风险指标"""
    # VaR（在险价值）
    var_95 = return_data.quantile(0.05)
    var_99 = return_data.quantile(0.01)
    
    # CVaR（条件在险价值）
    cvar_95 = return_data[return_data <= var_95].mean()
    cvar_99 = return_data[return_data <= var_99].mean()
    
    # 最大回撤
    cumulative = (1 + return_data).cumprod()
    running_max = cumulative.expanding().max()
    drawdown = (cumulative - running_max) / running_max
    max_drawdown = drawdown.min()
    
    # 夏普比率
    excess_return = return_data.mean() * 252 - risk_free_rate
    sharpe = excess_return / (return_data.std() * np.sqrt(252))
    
    return {
        'var_95': var_95,
        'var_99': var_99,
        'cvar_95': cvar_95,
        'cvar_99': cvar_99,
        'max_drawdown': max_drawdown,
        'sharpe_ratio': sharpe
    }

print("风险指标计算函数定义完成")

In [ ]:
# 4.4 年化波动率分布分析

def analyze_volatility_distribution(volatility_data, col_name=None):
    """分析年化波动率分布"""
    # 查找波动率列
    if col_name is None:
        possible_names = ['年华波动率', '年化波动率', '波动率']
        for col in volatility_data.columns:
            for name in possible_names:
                if name in str(col):
                    col_name = col
                    break
            if col_name:
                break
    
    if col_name is None:
        print("未找到年化波动率列")
        return None
    
    # 提取波动率数据
    data = volatility_data[col_name].dropna()
    
    # 基础统计
    stats = {
        'count': len(data),
        'min': data.min(),
        'max': data.max(),
        'mean': data.mean(),
        'median': data.median(),
        'std': data.std()
    }
    
    # 确定分组数量（Sturges规则）
    n = len(data)
    sturges_bins = int(np.ceil(np.log2(n) + 1))
    scott_bins = int(np.ceil((data.max() - data.min()) / 
                            (3.5 * data.std() / (n ** (1/3)))))
    optimal_bins = max(10, min(30, int((sturges_bins + scott_bins) / 2)))
    
    return {
        'data': data,
        'col_name': col_name,
        'stats': stats,
        'optimal_bins': optimal_bins
    }

print("波动率分布分析函数定义完成")

---
## 5. 执行与测试

主流程执行和结果验证。

In [ ]:
# 5.1 主流程执行

def main_analysis():
    """主分析流程"""
    print("=" * 60)
    print("理财产品权益配置分析")
    print("=" * 60)
    
    # Step 1: 数据采集
    print("\n[Step 1] 数据采集...")
    try:
        engine = get_database_engine()
        product_info = get_product_info(engine)
        print(f"获取产品信息: {len(product_info)} 条")
        
        nav_data = get_nav_data(engine=engine)
        print(f"获取净值数据: {len(nav_data)} 条")
        
        allocation_data = get_allocation_data(engine=engine)
        print(f"获取配置数据: {len(allocation_data)} 条")
    except Exception as e:
        print(f"数据库连接失败，使用模拟数据: {e}")
        # 使用模拟数据
        product_info = pd.DataFrame({
            'product_id': ['P001', 'P002', 'P003'],
            'product_name': ['产品A', '产品B', '产品C'],
            'risk_level': ['R2', 'R3', 'R4'],
            'investment_type': ['fixed_income', 'mixed', 'equity']
        })
        
    # Step 2: 数据处理
    print("\n[Step 2] 数据处理...")
    product_info = classify_products(product_info)
    print(f"产品分类完成，列: {list(product_info.columns)}")
    
    # Step 3: 权益配置分析
    print("\n[Step 3] 权益配置分析...")
    print("权益配置比例限制:")
    for key, value in WEALTH_PRODUCT_CONFIG.items():
        print(f"  - {key}: {value}")
    
    print("\n分析完成！")
    return product_info

# 执行主流程
result = main_analysis()

In [ ]:
# 5.2 波动率分析示例

def run_volatility_analysis(file_path=None):
    """运行波动率分析"""
    print("=" * 60)
    print("年化波动率分布分析")
    print("=" * 60)
    
    if file_path is None:
        print("\n未指定数据文件，使用模拟数据进行演示...")
        # 生成模拟数据
        np.random.seed(42)
        n_samples = 1000
        simulated_volatility = np.abs(np.random.normal(0.05, 0.03, n_samples))
        simulated_volatility = np.clip(simulated_volatility, 0, 0.3)
        
        volatility_df = pd.DataFrame({
            '年化波动率': simulated_volatility
        })
    else:
        volatility_df = load_excel_data(file_path, '年化波动率全量')
    
    if volatility_df is not None:
        analysis_result = analyze_volatility_distribution(volatility_df)
        if analysis_result:
            print(f"\n波动率统计信息:")
            for key, value in analysis_result['stats'].items():
                print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")
            print(f"\n建议分组数: {analysis_result['optimal_bins']}")
    
    return volatility_df

# 运行分析
volatility_result = run_volatility_analysis()

In [ ]:
# 5.3 结果验证

def validate_results():
    """验证分析结果"""
    print("=" * 60)
    print("结果验证")
    print("=" * 60)
    
    validation_results = []
    
    # 验证配置
    validation_results.append({
        '检查项': '配置参数完整性',
        '状态': '通过' if len(WEALTH_PRODUCT_CONFIG) == 5 else '失败',
        '详情': f'配置项数量: {len(WEALTH_PRODUCT_CONFIG)}'
    })
    
    # 验证风险等级
    validation_results.append({
        '检查项': '风险等级配置',
        '状态': '通过' if len(RISK_CLASSES) == 5 else '失败',
        '详情': f'风险等级: {list(RISK_CLASSES.keys())}'
    })
    
    # 验证投资类型
    validation_results.append({
        '检查项': '投资类型配置',
        '状态': '通过' if len(INVESTMENT_TYPES) == 3 else '失败',
        '详情': f'投资类型: {list(INVESTMENT_TYPES.keys())}'
    })
    
    # 显示验证结果
    validation_df = pd.DataFrame(validation_results)
    print(validation_df.to_string(index=False))
    
    return validation_df

# 执行验证
validation_result = validate_results()

---
## 6. 可视化

生成配置比例图、收益对比图、风险分析图。

In [ ]:
# 6.1 波动率分布可视化

def plot_volatility_distribution(data, optimal_bins=20):
    """绘制波动率分布图"""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('银行理财产品年化波动率分布分析', fontsize=16, fontweight='bold')
    
    # 1. 直方图
    axes[0, 0].hist(data, bins=optimal_bins, alpha=0.7, color='steelblue', edgecolor='black')
    axes[0, 0].axvline(data.mean(), color='red', linestyle='--', label=f'均值: {data.mean():.4f}')
    axes[0, 0].axvline(data.median(), color='green', linestyle='--', label=f'中位数: {data.median():.4f}')
    axes[0, 0].set_xlabel('年化波动率')
    axes[0, 0].set_ylabel('频数')
    axes[0, 0].set_title('频数分布直方图')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. 密度图
    data.plot.density(ax=axes[0, 1], color='darkblue', linewidth=2)
    axes[0, 1].fill_between(data.plot.density().get_xdata(), 
                           data.plot.density().get_ydata(), 
                           alpha=0.3, color='lightblue')
    axes[0, 1].set_xlabel('年化波动率')
    axes[0, 1].set_ylabel('密度')
    axes[0, 1].set_title('核密度估计图')
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. 箱线图
    axes[1, 0].boxplot(data, vert=True, patch_artist=True,
                      boxprops=dict(facecolor='lightgreen', alpha=0.7))
    axes[1, 0].set_ylabel('年化波动率')
    axes[1, 0].set_title('箱线图')
    axes[1, 0].grid(True, alpha=0.3)
    
    # 4. 散点图
    y_jitter = np.random.normal(0, 0.02, len(data))
    axes[1, 1].scatter(data, y_jitter, alpha=0.5, s=20, c='coral')
    axes[1, 1].set_xlabel('年化波动率')
    axes[1, 1].set_ylabel('分布密度（抖动）')
    axes[1, 1].set_title('点状分布图')
    axes[1, 1].set_ylim(-0.1, 0.1)
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return fig

# 使用模拟数据演示
np.random.seed(42)
demo_data = pd.Series(np.abs(np.random.normal(0.05, 0.03, 500)))
demo_data = demo_data.clip(0, 0.2)

# plot_volatility_distribution(demo_data)

In [ ]:
# 6.2 权益配置比例图

def plot_equity_allocation(allocation_result):
    """绘制权益配置比例图"""
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # 饼图 - 权益类型分布
    equity_labels = list(EQUITY_TYPES.values())
    equity_values = [0.4, 0.25, 0.2, 0.1, 0.05]  # 示例数据
    
    axes[0].pie(equity_values, labels=equity_labels, autopct='%1.1f%%',
               colors=plt.cm.Set3(np.linspace(0, 1, len(equity_labels))))
    axes[0].set_title('权益资产类型分布')
    
    # 柱状图 - 按风险等级的权益配置
    risk_levels = list(RISK_CLASSES.values())
    equity_ratios = [0.05, 0.15, 0.30, 0.50, 0.70]  # 示例数据
    
    bars = axes[1].bar(risk_levels, equity_ratios, color='steelblue', edgecolor='black')
    axes[1].set_xlabel('风险等级')
    axes[1].set_ylabel('权益配置比例')
    axes[1].set_title('各风险等级权益配置比例')
    axes[1].set_ylim(0, 1)
    
    # 添加数值标签
    for bar, ratio in zip(bars, equity_ratios):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                    f'{ratio:.0%}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()
    
    return fig

print("权益配置可视化函数定义完成")

---
## 7. 工具函数

可复用的辅助函数。

In [ ]:
# 7.1 日期处理工具

import datetime

def get_quarter_dates(year, quarter):
    """获取指定季度的起止日期"""
    quarter_starts = {
        1: datetime.date(year, 1, 1),
        2: datetime.date(year, 4, 1),
        3: datetime.date(year, 7, 1),
        4: datetime.date(year, 10, 1)
    }
    quarter_ends = {
        1: datetime.date(year, 3, 31),
        2: datetime.date(year, 6, 30),
        3: datetime.date(year, 9, 30),
        4: datetime.date(year, 12, 31)
    }
    return quarter_starts[quarter], quarter_ends[quarter]

def get_trading_days(start_date, end_date):
    """获取交易日列表（简化版，排除周末）"""
    dates = pd.date_range(start_date, end_date, freq='B')  # 工作日
    return dates

print("日期处理工具定义完成")

In [ ]:
# 7.2 数据导出工具

def export_to_excel(data_dict, file_path):
    """导出数据到Excel文件"""
    with pd.ExcelWriter(file_path, engine='openpyxl') as writer:
        for sheet_name, df in data_dict.items():
            df.to_excel(writer, sheet_name=sheet_name, index=False)
    print(f"数据已导出到: {file_path}")

def export_analysis_report(results, file_path):
    """导出分析报告"""
    report_lines = [
        "=" * 60,
        "理财产品权益配置分析报告",
        "=" * 60,
        f"\n生成时间: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        "\n" + "-" * 40,
        "分析结果摘要",
        "-" * 40
    ]
    
    for key, value in results.items():
        if isinstance(value, (int, float)):
            report_lines.append(f"{key}: {value:.4f}")
        else:
            report_lines.append(f"{key}: {value}")
    
    with open(file_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(report_lines))
    
    print(f"报告已导出到: {file_path}")

print("数据导出工具定义完成")

In [ ]:
# 7.3 统计分析工具

def calculate_statistics(data):
    """计算基础统计量"""
    return {
        'count': len(data),
        'mean': np.mean(data),
        'median': np.median(data),
        'std': np.std(data),
        'min': np.min(data),
        'max': np.max(data),
        'q25': np.percentile(data, 25),
        'q75': np.percentile(data, 75),
        'skewness': pd.Series(data).skew(),
        'kurtosis': pd.Series(data).kurtosis()
    }

def calculate_percentiles(data, percentiles=[1, 5, 10, 25, 50, 75, 90, 95, 99]):
    """计算分位数"""
    return {f'p{p}': np.percentile(data, p) for p in percentiles}

print("统计分析工具定义完成")

In [ ]:
# 7.4 配置验证工具

def validate_equity_allocation(allocation_ratio, risk_level):
    """验证权益配置是否符合限制"""
    config = WEALTH_PRODUCT_CONFIG
    
    warnings_list = []
    
    # 检查最小/最大比例
    if allocation_ratio < config['min_equity_ratio']:
        warnings_list.append(f"权益比例 {allocation_ratio:.2%} 低于最小限制 {config['min_equity_ratio']:.2%}")
    if allocation_ratio > config['max_equity_ratio']:
        warnings_list.append(f"权益比例 {allocation_ratio:.2%} 超过最大限制 {config['max_equity_ratio']:.2%}")
    
    # 检查风险等级与配置比例匹配度
    risk_equity_limits = {
        'R1': (0, 0.1),
        'R2': (0, 0.2),
        'R3': (0.1, 0.4),
        'R4': (0.2, 0.6),
        'R5': (0.4, 0.8)
    }
    
    if risk_level in risk_equity_limits:
        min_limit, max_limit = risk_equity_limits[risk_level]
        if not (min_limit <= allocation_ratio <= max_limit):
            warnings_list.append(f"风险等级 {risk_level} 建议权益比例在 {min_limit:.0%}-{max_limit:.0%} 范围内")
    
    return {
        'is_valid': len(warnings_list) == 0,
        'warnings': warnings_list
    }

print("配置验证工具定义完成")

---
## 总结

本Notebook完成了理财产品权益配置的完整分析流程：

1. **环境配置**：导入依赖并设置参数
2. **数据获取**：定义数据库和Excel数据读取函数
3. **数据处理**：数据清洗、产品分类、配置比例计算
4. **核心逻辑**：权益配置分析、收益计算、风险指标计算
5. **执行与测试**：主流程执行和结果验证
6. **可视化**：分布图、配置图等
7. **工具函数**：日期处理、数据导出、统计分析等

### 核心价值
- 产品分析：按风险等级、投资类型分类
- 配置分析：权益资产配置比例分析
- 收益分析：收益率、波动率计算
- 风险管控：VaR、CVaR、最大回撤、夏普比率